In [ ]:
import os
import re
import pdfplumber
import pandas as pd
import nltk

# Change this directory to the location of the files on your computer
PDF_FOLDER = "/Users/emilymoore/Downloads/DS 4002 Project 2/meeting_mins"

# Define a list of uncertainty-related keywords
# These capture forward-looking language, risk, ambiguity, or policy hesitation
UNCERTAINTY_WORDS = [
    "uncertain","uncertainty","risk","risks","may","might",
    "could","potential","possibly","unknown","unclear",
    "concern","concerns","challenge","challenges",
    "pending","depend","depends"
]

# Define housing-related keywords
# These ensure we focus specifically on housing-policy discussions
HOUSING_WORDS = [
    "housing","affordable housing","zoning",
    "development","residential","rent",
    "apartment","units","subsidy"
]

# Function to extract full text from a .pdf file
def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# Function to count occurrences of words in a text
def count_words(text, word_list):
    lower_text = text.lower()
    return sum(lower_text.count(word.lower()) for word in word_list)

# Function to count total words in a text
def count_total_words(text):
    words = re.findall(r"\b\w+\b", text)
    return len(words)

# Function to extract meeting start and end times using regex
def get_meeting_times(text):
    # Regex pattern to capture meeting start time
    # Added \s* everywhere to be safe with line breaks in text files
    start_pattern = r"calls\s+meeting\s+to\s+order\s*(?:at)?\s*((\d{1,2}:\d{2})\s*([AaPp][Mm])?)"
    end_pattern = r"adjourned\s*(?:at)?\s*((\d{1,2}:\d{2})\s*([AaPp][Mm])?)"
    # Search for patterns in the full text
    start_match = re.search(start_pattern, text, re.IGNORECASE)
    end_match = re.search(end_pattern, text, re.IGNORECASE)
    # Return extracted times if found; otherwise None
    return {
        "start_time": start_match.group(1).strip() if start_match else None,
        "end_time": end_match.group(1).strip() if end_match else None
    }
results = [] # Initialize an empty list to store results for each meeting file

# Loop through PDF files
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"): # Process only .pdf files
        path = os.path.join(PDF_FOLDER, filename)
        
        full_text = extract_text_from_pdf(path) # Extract full text
        times = get_meeting_times(full_text) # Extract meeting start and end times

        housing_mentions = count_words(full_text, HOUSING_WORDS) # Extract housing words in text
        uncertainty_mentions = count_words(full_text, UNCERTAINTY_WORDS) # Extract uncertainty terms in text

        total_words = count_total_words(full_text) # Extract total number of words in text

        sentences = nltk.sent_tokenize(full_text.lower()) # Tokenize text into sentences using NLTK
        # Convert to lowercase to standardize matching

        # Identify sentences that contain BOTH a housing-related word AND an uncertainty-related word
        # This captures "housing uncertainty density"
        housing_uncertainty_sentences = [
            s for s in sentences
            if any(h in s for h in HOUSING_WORDS)
            and any(u in s for u in UNCERTAINTY_WORDS)
        ]
    # Append structured results for this file
        results.append({
            "file_name": filename,
            "start_time": times["start_time"],
            "end_time": times["end_time"],
            "housing_mentions": housing_mentions,
            "uncertainty_mentions": uncertainty_mentions,
            "housing_uncertainty_sentences": len(housing_uncertainty_sentences),
            "minutes_word_length": total_words
        })


df_mins = pd.DataFrame(results) # Convert list of dictionaries into a DataFrame
print(df_mins) # Print the structured output to verify extraction worked

# Save results
df_mins.to_csv(os.path.join(PDF_FOLDER, "housing_uncertainty.csv"), index=False)

In [ ]:
# Create updated csv with uncertainty score

# Change the path to match your directory
df_mins_with_score = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 2/meeting_mins/housing_uncertainty.csv')

# Extract the month/year from file_name for chronological sorting
def extract_date(filename):
    match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', filename)
    if match:
        month = int(match.group(1))
        year = match.group(3)
        return f"{year}-{month:02d}"
    return None

df_mins_with_score['month_year'] = df_mins_with_score['file_name'].apply(extract_date)

# Calculate the Uncertainty Score
# Density of housing-uncertainty sentences per housing mention
df_mins_with_score['uncertainty_score'] = df_mins_with_score['housing_uncertainty_sentences'] / df_mins_with_score['housing_mentions']

# Handle cases where housing_mentions is 0 to avoid errors
df_mins_with_score['uncertainty_score'] = df_mins_with_score['uncertainty_score'].fillna(0)

# Sort by date and save
df_mins_with_score = df_mins_with_score.sort_values('month_year')
df_mins_with_score.to_csv('/Users/emilymoore/Downloads/DS 4002 Project 2/meeting_mins/housing_uncertainty_with_score.csv', index=False)

print("Updated the CSV to '/Users/emilymoore/Downloads/DS 4002 Project 2/meeting_mins/housing_uncertainty_with_score.csv")